Target definition (most standard for finance ML): We’ll predict next-day log return

Load data + choose feature columns

In [1]:
import pandas as pd
import numpy as np

PATH = "../data/processed/features/BTC_features.csv"
df = pd.read_csv(PATH)

df["date"] = pd.to_datetime(df["date"])
df = df.sort_values("date").reset_index(drop=True)

# We intentionally exclude market_cap for v1 because it has many NaNs.
FEATURES = [
    "log_ret_1d", "log_ret_5d", "log_ret_10d",
    "vol_7d", "vol_30d",
    "risk_adj_ret_1d", "vol_ratio_7d_30d",
    "drawdown_30d"
]

print(df[FEATURES].isna().sum())
df.head()


log_ret_1d          0
log_ret_5d          0
log_ret_10d         0
vol_7d              0
vol_30d             0
risk_adj_ret_1d     0
vol_ratio_7d_30d    0
drawdown_30d        0
dtype: int64


,date,open,high,low,close,volume,ticker,market_cap,log_ret_1d,log_ret_5d,log_ret_10d,vol_7d,vol_30d,risk_adj_ret_1d,vol_ratio_7d_30d,drawdown_30d
0,2020-01-31,9508.313477,9521.706055,9230.776367,9350.529297,29432489719,BTC-USD,NaN,-0.016805,0.084039,0.066849,0.024991,0.028110,-0.597831,0.889034,-0.016665
1,2020-02-01,9346.357422,9439.323242,9313.239258,9392.875000,25922656496,BTC-USD,NaN,0.004518,0.052797,0.078829,0.023294,0.027147,0.166442,0.858044,-0.012211
2,2020-02-02,9389.820312,9468.797852,9217.824219,9344.365234,30835736946,BTC-USD,NaN,-0.005178,-0.001521,0.105766,0.024041,0.026178,-0.197799,0.918398,-0.017313
3,2020-02-03,9344.683594,9540.372070,9248.633789,9293.521484,30934096509,BTC-USD,NaN,-0.005456,-0.002483,0.095692,0.022204,0.026292,-0.207515,0.844516,-0.022660
4,2020-02-04,9292.841797,9331.265625,9112.811523,9180.962891,29893183716,BTC-USD,NaN,-0.012185,-0.035106,0.092735,0.012202,0.026507,-0.459713,0.460328,-0.034497


Create the target (next-day return)

In [2]:
# Target: next-day log return
df["target_log_ret_1d"] = df["log_ret_1d"].shift(-1)

# Drop the last row (target is unknown there)
df_model = df.dropna(subset=["target_log_ret_1d"]).reset_index(drop=True)

print(df_model[["date", "log_ret_1d", "target_log_ret_1d"]].tail(5))
print("Rows for modeling:", len(df_model))


           date  log_ret_1d  target_log_ret_1d
2195 2026-02-03   -0.039600          -0.035171
2196 2026-02-04   -0.035171          -0.152334
2197 2026-02-05   -0.152334           0.118003
2198 2026-02-06    0.118003          -0.018213
2199 2026-02-07   -0.018213           0.023808
Rows for modeling: 2200


Time-series split (no shuffle!)

In [3]:
split_idx = int(len(df_model) * 0.8)

train = df_model.iloc[:split_idx].copy()
test  = df_model.iloc[split_idx:].copy()

X_train = train[FEATURES]
y_train = train["target_log_ret_1d"]

X_test  = test[FEATURES]
y_test  = test["target_log_ret_1d"]

print("Train:", train["date"].min(), "→", train["date"].max(), "n=", len(train))
print("Test :", test["date"].min(),  "→", test["date"].max(),  "n=", len(test))


Train: 2020-01-31 00:00:00 → 2024-11-24 00:00:00 n= 1760
Test : 2024-11-25 00:00:00 → 2026-02-07 00:00:00 n= 440


Train a baseline ML model

We’ll start with a Ridge regression baseline: fast, stable, interpretable, strong starting point for returns

In [4]:
import sklearn
print("sklearn version:", sklearn.__version__)


sklearn version: 1.8.0


Train Ridge model (with scaling)

In [5]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge

ridge = Pipeline([
    ("scaler", StandardScaler()),
    ("model", Ridge(alpha=1.0))
])

ridge.fit(X_train, y_train)

pred_test = ridge.predict(X_test)
pred_train = ridge.predict(X_train)

pred_test[:5], pred_train[:5]


(array([ 0.00303813, -0.00067403, -0.00268104,  0.00036382, -0.00035059]),
 array([0.00359033, 0.00240206, 0.00025874, 0.00044262, 0.00024244]))

Evaluate (MSE, MAE, correlation, directional accuracy):

Returns are noisy; accuracy is not like classification. We evaluate multiple ways.

In [6]:
from sklearn.metrics import mean_squared_error, mean_absolute_error

def directional_accuracy(y_true, y_pred):
    return (np.sign(y_true) == np.sign(y_pred)).mean()

mse = mean_squared_error(y_test, pred_test)
mae = mean_absolute_error(y_test, pred_test)
corr = np.corrcoef(y_test, pred_test)[0, 1]
da = directional_accuracy(y_test.values, pred_test)

print("Test MSE:", mse)
print("Test MAE:", mae)
print("Test Corr:", corr)
print("Directional Accuracy:", da)


Test MSE: 0.0005710717938018047
Test MAE: 0.016710736982329343
Test Corr: 0.08895622091538845
Directional Accuracy: 0.5295454545454545


In [7]:
coef = ridge.named_steps["model"].coef_
coef_df = pd.DataFrame({"feature": FEATURES, "coef": coef}).sort_values("coef", ascending=False)

coef_df


,feature,coef
5,risk_adj_ret_1d,0.009564
1,log_ret_5d,0.002597
3,vol_7d,0.002563
2,log_ret_10d,-0.000053
7,drawdown_30d,-0.000377
4,vol_30d,-0.001026
6,vol_ratio_7d_30d,-0.002128
0,log_ret_1d,-0.011738


Overfitting check: Compare train vs test performance

In [8]:
from sklearn.metrics import mean_squared_error, mean_absolute_error
import numpy as np

def evaluate(y_true, y_pred, label):
    mse = mean_squared_error(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    corr = np.corrcoef(y_true, y_pred)[0, 1]
    da = (np.sign(y_true) == np.sign(y_pred)).mean()

    print(f"{label}")
    print("  MSE:", mse)
    print("  MAE:", mae)
    print("  Corr:", corr)
    print("  Dir Acc:", da)
    print()

evaluate(y_train, pred_train, "TRAIN")
evaluate(y_test, pred_test, "TEST")


TRAIN
  MSE: 0.0011557766359488943
  MAE: 0.022618198764237103
  Corr: 0.14151003935129428
  Dir Acc: 0.5176136363636363

TEST
  MSE: 0.0005710717938018047
  MAE: 0.016710736982329343
  Corr: 0.08895622091538845
  Dir Acc: 0.5295454545454545



The model is regularized and stable
It generalizes at least as well as it fits
No memorization, no leakage

Walk-forward CV with expanding window

In [9]:
from sklearn.model_selection import TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
import numpy as np

tscv = TimeSeriesSplit(n_splits=5)

corrs = []
das = []

for fold, (train_idx, val_idx) in enumerate(tscv.split(df_model)):
    train = df_model.iloc[train_idx]
    val   = df_model.iloc[val_idx]

    X_tr = train[FEATURES]
    y_tr = train["target_log_ret_1d"]

    X_val = val[FEATURES]
    y_val = val["target_log_ret_1d"]

    model = Pipeline([
        ("scaler", StandardScaler()),
        ("model", Ridge(alpha=1.0))
    ])

    model.fit(X_tr, y_tr)
    preds = model.predict(X_val)

    corr = np.corrcoef(y_val, preds)[0, 1]
    da = (np.sign(y_val) == np.sign(preds)).mean()

    corrs.append(corr)
    das.append(da)

    print(f"Fold {fold+1}: Corr={corr:.3f}, DirAcc={da:.3f}")

print("\nAverage Corr:", np.nanmean(corrs))
print("Average Dir Acc:", np.mean(das))


Fold 1: Corr=0.077, DirAcc=0.503
Fold 2: Corr=0.130, DirAcc=0.475
Fold 3: Corr=0.002, DirAcc=0.486
Fold 4: Corr=0.062, DirAcc=0.514
Fold 5: Corr=0.105, DirAcc=0.514

Average Corr: 0.07528187149085189
Average Dir Acc: 0.49836065573770494


Correlation

Positive on 4 out of 5 folds

One fold near zero (this is normal — markets change)

Average ≈ 0.07–0.08 is solid for daily crypto returns

In practice:

Many production alpha signals live permanently in the 0.03–0.10 correlation range.

So your signal is real but weak, which is exactly what we want before adding probabilistic structure.

✔ Directional accuracy

Around 50% on average

Slightly above in later folds

One fold below 50% (expected)

This tells us something important:

The model is not a directional oracle — and that’s okay.

In modern quant systems:

Directional accuracy alone is not the goal

Risk-aware decisions and distribution tails matter more